# Tree Baseline (LightGBM)

In [1]:
# optional if you have trouble running the environments
import sys
sys.path.insert(0, "../src")

# regular imports
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import MultiLabelBinarizer
from remy.paths import PROJECT_ROOT
DATA = PROJECT_ROOT / "data" / "raw"

SEED = 42

## Load data

In [2]:
recipes = pd.read_parquet(DATA / "recipes.parquet", columns=["id", "name", "tags"])
interactions = pd.read_parquet(
    DATA / "interactions.parquet", columns=["user_id", "recipe_id", "date", "rating"]
)
print("Recipes:", recipes.shape, "Interactions:", interactions.shape, "Unique users:", interactions["user_id"].nunique())

Recipes: (231637, 3) Interactions: (1132367, 4) Unique users: 226570


## Train/val/test split

Split by date (temporally) in a 80/10/10 breakdown. This is deterministic, so we can be sure of the same splits every time. 

In [3]:
t_val, t_test = interactions["date"].quantile([0.8, 0.9])
train = interactions[interactions["date"] < t_val]
val = interactions[(interactions["date"] >= t_val) & (interactions["date"] < t_test)]
test = interactions[interactions["date"] >= t_test]
print(f"train: {len(train):,} | val: {len(val):,} | test: {len(test):,}")
print(f"val from {t_val.date()}, test from {t_test.date()}")

train: 905,786 | val: 113,312 | test: 113,269
val from 2011-12-27, test from 2014-02-25


## Recipe tag features

Multi-hot encode each recipe's tags into a binary vector, dropping rare tags to keep the feature space small.

In [4]:
MIN_TAG_COUNT = 100  # drop tags that appear on very few recipes

tag_counts = recipes["tags"].explode().value_counts()
vocab = tag_counts[tag_counts >= MIN_TAG_COUNT].index.tolist()
vocab_set = set(vocab)
print(f"{len(tag_counts)} distinct tags, keeping {len(vocab)} with >= {MIN_TAG_COUNT} recipes")

mlb = MultiLabelBinarizer(classes=vocab, sparse_output=True)
recipe_tags = mlb.fit_transform(
    [[t for t in tags if t in vocab_set] for tags in recipes["tags"]]
).astype(np.float32).tocsr()
row_of_recipe = pd.Series(np.arange(len(recipes)), index=recipes["id"])
print("recipe x tag matrix:", recipe_tags.shape)

552 distinct tags, keeping 412 with >= 100 recipes


recipe x tag matrix: (231637, 412)


## Training data

Each row's features are just that recipe's tag vector — no user features this time.

In [5]:
train_rated = train[train["rating"] > 0]

X_train = recipe_tags[row_of_recipe.reindex(train_rated["recipe_id"]).to_numpy()]
y_train = (train_rated["rating"] >= 4).to_numpy()
print("X_train:", X_train.shape, "positive rate:", round(y_train.mean(), 3))

X_train: (873201, 412) positive rate: 0.942


## Train the model

A single LightGBM classifier, no tuning — `val` is used for eval only, not for fitting.

In [6]:
model = lgb.LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1)
model.fit(X_train, y_train)

,n_estimators,200
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


## Top 10 recipes by predicted score

No personalization now, so — like `popularity_baseline.ipynb` — this is one ranking shown to everyone. Raw model score alone favors obscure recipes with "good-looking" tags but almost no real interaction history, so blend the score with `log1p(train interaction count)` instead of ranking on score alone — popularity acts as a confidence weight on the content score.

In [7]:
train_counts = train["recipe_id"].value_counts().reindex(recipes["id"], fill_value=0)

scores = pd.Series(model.predict_proba(recipe_tags)[:, 1], index=recipes["id"])
blended_scores = scores * np.log1p(train_counts.to_numpy())

top10_ids = blended_scores.nlargest(10).index
top10_set = set(top10_ids)
recipes.set_index("id").loc[top10_ids, "name"]

id
39087                           creamy cajun chicken pasta
32204                  whatever floats your boat  brownies
67256     best ever banana cake with cream cheese frosting
27208                           to die for crock pot roast
22782                     jo mama s world famous spaghetti
69173     kittencal s italian melt in your mouth meatballs
28148                      oven fried chicken chimichangas
89204    crock pot chicken with black beans   cream cheese
54257              yes  virginia there is a great meatloaf
25885                                  banana banana bread
Name: name, dtype: string

## Evaluation

**Hit Rate@10** and **Recall@10**, each computed for two definitions of "positive": rating ≥ 4, and rating = 5 (a harder bar). Ratings of 0 (no star given) are excluded for evaluation only. 

- **Hit Rate@10**: did any of a user's positive-rated eval recipes land in our top 10, averaged over every user with at least one rated eval non-zero interaction
- **Recall@10**: what fraction of a user's positive-rated test recipes landed in our top 10, averaged over every user with at least one rated eval non-zero interaction

In [8]:
eval_rated = val[val["rating"] > 0]
users_with_signal = eval_rated["user_id"].unique()


def evaluate(min_rating):
    """Hit Rate@10 and Recall@10 for positive = rating >= min_rating."""
    positive = eval_rated[eval_rated["rating"] >= min_rating]
    positive_items_by_user = positive.groupby("user_id")["recipe_id"].agg(set)
    hits_by_user = positive_items_by_user.apply(lambda items: len(items & top10_set))

    # hit rate: every user with a rated eval interaction, negative-only users count as a miss
    hit_rate = (hits_by_user > 0).reindex(users_with_signal, fill_value=False).mean()

    # recall: only users with >=1 positive item, since it's undefined otherwise
    recall = (hits_by_user / positive_items_by_user.apply(len)).mean()

    return hit_rate, recall

In [9]:
rows = []
for min_rating, label in [(4, "rating >= 4"), (5, "rating = 5")]:
    hit_rate, recall = evaluate(min_rating)
    rows.append({"positive threshold": label, "Hit Rate@10": hit_rate, "Recall@10": recall})

pd.DataFrame(rows)

,positive threshold,Hit Rate@10,Recall@10
0,rating >= 4,0.029304,0.019257
1,rating = 5,0.026051,0.020071
